### Chatbot Evaluation

1. Create a dataset with ques and expected answers
2. Get output from model for input
3. Check how well llm performed using evaluators
4. Choose llm which performs best

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"


In [5]:
from langsmith import Client

client=Client()

dataset_name = "Chatbots Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)


{'example_ids': ['a020e58f-042d-4594-a80e-89599bbd6ec9',
  'f9a8a74a-cef9-46e4-9e23-43dd78c0e8ba',
  '4954cd39-eaa1-4d70-9a00-dbcad854e41b',
  '53528532-6315-4611-ba4c-f087dfa52581',
  'de473ac6-e706-43e9-a382-c12141c5f747'],
 'count': 5,
 'as_of': '2026-06-01T16:52:08.587819454Z'}

In [9]:
import groq
from langsmith import traceable

# Use the standard unwrapped Groq client
groq_client = groq.Groq()

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

# Simply add the @traceable decorator here
@traceable
def correctness(inputs:dict,outputs:dict, reference_outputs:dict)->bool:
      user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
      response = groq_client.chat.completions.create(
            model="qwen/qwen3-32b",
            temperature=0,
            messages=[
                  {"role":"system","content":eval_instructions},
                  {"role":"user","content":user_content}
            ]
      ).choices[0].message.content

      return response.strip().upper() == "CORRECT"


In [12]:


def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

In [10]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."
def my_app(question: str, model: str = "qwen/qwen3-32b", instructions: str = default_instructions) -> str:
    return groq_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    ).choices[0].message.content

In [11]:
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [13]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, 
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="groq-qwen3-mini-chatbot"
)

/Users/krishrohilla/Desktop/MindzKonnected/Tutorial/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'groq-qwen3-mini-chatbot-40b056af' at:
https://smith.langchain.com/o/52062f21-4523-41d3-bef2-e56da297d460/datasets/459f6f07-74bc-433a-a659-c39b179fa6b8/compare?selectedSessions=709260cb-7dc4-4996-8eeb-6c93dbbb69c0




5it [00:10,  2.07s/it]
